In [ ]:
# =====================================================================
# 05 - VALIDATION: does the genomic fertility EBV predict real fertility?
# ---------------------------------------------------------------------
# WHAT WE EVALUATE:
#   Whether Daughter Fertility (DF), a genomic breeding value, predicts the
#   actual reproductive performance (No. Services) of the cows in this herd.
#
# H0: rho(DF, No. Services) = 0  -> the genomic evaluation carries no
#     information about real fertility in this herd.
# H1: rho < 0  -> higher genetic merit for fertility means fewer services.
# DECISION RULE: reject H0 at p < 0.05 (Pearson).
#
# BENCHMARK (this is the point):
#   For an animal's OWN phenotype, quantitative genetics predicts
#       cor(EBV, P) = accuracy x h,   where h = sqrt(heritability).
#   Fertility heritability is low (h2 ~ 0.02-0.05 in the literature; verify
#   against Lactanet's published value), so h ~ 0.14-0.22. Even a PERFECT
#   EBV could not exceed ~0.22. The question is therefore NOT "is r large?"
#   but "is r consistent with theory?"
#
# METHODOLOGICAL CONTROLS:
#   1. CENSORING: a cow with 3 services already pregnant differs from one
#      with 3 services still open (she will need more). Only PREGNANT cows
#      are analysed (n = 305).
#   2. PARITY: services vary with lactation number -> both variables are
#      residualised on Lact. Number.
# =====================================================================
import os, pandas as pd, numpy as np, matplotlib.pyplot as plt
from scipy import stats
from math import sqrt
os.makedirs("outputs", exist_ok=True)
plt.rcParams.update({"figure.dpi":110,"savefig.bbox":"tight","figure.facecolor":"white",
 "axes.titlesize":13,"axes.titleweight":"bold","axes.spines.top":False,
 "axes.spines.right":False,"axes.grid":True,"grid.color":"#E5E7EB",
 "axes.axisbelow":True,"font.size":10.5})

df = pd.read_excel("../Master_Database_Consolidated_Final.xlsx")
df.columns = [c.strip() for c in df.columns]
for c in ["DF","No. Services","Lact. Number","Genomic Inb. %"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

def resid(y, X):
    X = np.column_stack([np.ones(len(X)), np.asarray(X, float)])
    b, *_ = np.linalg.lstsq(X, np.asarray(y, float), rcond=None)
    return np.asarray(y, float) - X @ b

# ---- Step 1: naive (censored) ----
d_all = df[["DF","No. Services"]].dropna(); d_all = d_all[d_all["No. Services"] > 0]
r0, p0 = stats.pearsonr(d_all["DF"], d_all["No. Services"])

# ---- Step 2: remove censoring -> pregnant cows only ----
preg = df[(df["Preg Status"] == "P") & (df["No. Services"] > 0)].dropna(
    subset=["DF","No. Services","Lact. Number"])
r1, p1 = stats.pearsonr(preg["DF"], preg["No. Services"])

# ---- Step 3: adjust for parity ----
r2, p2 = stats.pearsonr(resid(preg["DF"], preg[["Lact. Number"]]),
                        resid(preg["No. Services"], preg[["Lact. Number"]]))

n = len(preg)
mde = np.tanh((1.96 + 0.84) / sqrt(n - 3))

print("=" * 74); print("VALIDATION OF THE GENOMIC FERTILITY EVALUATION"); print("=" * 74)
print(f"{'Model':38s} {'r':>8s} {'p':>9s} {'n':>5s}")
print(f"{'1. All cows (censored)':38s} {r0:+8.3f} {p0:9.4f} {len(d_all):5d}")
print(f"{'2. Pregnant only (censoring removed)':38s} {r1:+8.3f} {p1:9.4f} {n:5d}")
print(f"{'3. + adjusted for lactation number':38s} {r2:+8.3f} {p2:9.4f} {n:5d}")
print(f"\nPower: n = {n} -> minimum detectable |r| at 80% power = {mde:.3f}")
print(f"H0 {'REJECTED' if p2 < 0.05 else 'not rejected'}")

print("\n" + "=" * 74); print("IS THE RESULT CONSISTENT WITH THEORY?"); print("=" * 74)
print("  cor(EBV, own phenotype) = accuracy x h ,  h = sqrt(heritability)")
print(f"{'h2':>6s} {'h':>6s} {'r if acc=0.5':>13s} {'r if acc=0.7':>13s} {'implied acc':>12s}")
for h2 in [0.02, 0.03, 0.05, 0.10]:
    h = sqrt(h2)
    implied = abs(r2) / h
    print(f"{h2:6.2f} {h:6.3f} {0.5*h:13.3f} {0.7*h:13.3f} {implied:12.2f}")
print(f"\n  OBSERVED |r| = {abs(r2):.3f}")
print("  -> The EBV is NOT underperforming: it matches or exceeds the ceiling")
print("     that low heritability imposes. Even a perfect EBV could not exceed h.")

# ---- Inbreeding depression on the real phenotype ----
di = df[(df["Preg Status"] == "P") & (df["No. Services"] > 0)].dropna(
    subset=["Genomic Inb. %","No. Services","Lact. Number"])
ri, pi = stats.pearsonr(resid(di["Genomic Inb. %"], di[["Lact. Number"]]),
                        resid(di["No. Services"], di[["Lact. Number"]]))
print("\n" + "=" * 74); print("INBREEDING DEPRESSION ON THE REAL PHENOTYPE"); print("=" * 74)
print(f"  r(Genomic Inb. %, No. Services | lactation) = {ri:+.3f} (p={pi:.4f}, n={len(di)})")

# ---- FIGURE ----
fig, ax = plt.subplots(1, 2, figsize=(15, 5.5))
grp = preg.groupby(pd.qcut(preg["DF"], 4, labels=["Q1 (lowest DF)","Q2","Q3","Q4 (highest DF)"]),
                   observed=True)["No. Services"]
m, se = grp.mean(), grp.std()/np.sqrt(grp.count())
ax[0].bar(range(4), m.values, yerr=1.96*se.values, color=["#EF4444","#F59E0B","#84CC16","#10B981"],
          edgecolor="white", capsize=7, error_kw={"lw":2,"ecolor":"#374151"})
ax[0].set_xticks(range(4)); ax[0].set_xticklabels(m.index, rotation=15, ha="right")
ax[0].set_ylabel("Services to conception (mean ± 95% CI)")
ax[0].set_title(f"Higher genomic fertility merit → fewer real services\n(pregnant cows, n={n})")

steps = ["All cows\n(censored)", "Pregnant only", "+ parity\nadjusted"]
vals  = [abs(r0), abs(r1), abs(r2)]
ax[1].bar(steps, vals, color=["#9CA3AF","#60A5FA","#2563EB"], edgecolor="white")
ax[1].axhline(mde, ls="--", color="#EF4444", lw=2, label=f"Detection limit (r={mde:.3f})")
ax[1].axhline(sqrt(0.05), ls=":", color="#1F2937", lw=2,
              label=f"Theoretical ceiling if h²=0.05 (r={sqrt(0.05):.3f})")
ax[1].set_ylabel("|r| with No. Services")
ax[1].set_title("Fixing censoring and parity strengthens the signal")
ax[1].legend(frameon=False, fontsize=9)
plt.tight_layout(); plt.savefig("outputs/05_fertility_validation.png", dpi=150); plt.show()